In [3]:
# Библиотека
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind

In [31]:
data_audience = pd.read_excel('test_task.xlsx', sheet_name='Данные об аудитории')

In [32]:
data_audience.head()

,date,user_id,view_adverts
0,2023-11-11,8c020470-8461-11ed-83d0-552e8cc749d6,13
1,2023-11-18,5875f070-7b92-11ee-a6fb-8b298e83f4f7,14
2,2023-11-29,3c2d27c0-4fd6-11eb-b89f-2ffb31b67dd6,21
3,2023-11-29,234a96d0-ad16-11ed-a2e6-793ddfeeba1f,23
4,2023-11-29,4d07c180-644f-11eb-879c-b7c02edf4f37,12


In [36]:
data_audience.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16814 entries, 0 to 16813
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          16814 non-null  datetime64[ns]
 1   user_id       16814 non-null  object        
 2   view_adverts  16814 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 394.2+ KB


In [2]:
# 1. Во вкладке "Данные об аудитории" информация о пользователях, посетивших наше приложение в ноябре. Чему равен MAU продукта?

In [35]:
data_audience['date'].dt.month.unique()

array([11], dtype=int32)

In [37]:
data_audience['user_id'].nunique()

7639

In [ ]:
# 2. Используя вкладку "Данные об аудитории", посчитайте, чему будет равен DAU 

In [88]:
print(data_audience.groupby('date')['user_id'].nunique().mean())

560.4666666666667


In [42]:
# 3. Используя вкладку "Данные об аудитории", посчитайте, чему будет равен retention первого дня у пользователей, пришедших в продукт 1 ноября 

In [53]:
data_audience.groupby('date')['user_id'].nunique().head(2)

date
2023-11-01    623
2023-11-02    649
Name: user_id, dtype: int64

In [90]:
first_day_users = set(data_audience[data_audience['date'] == '2023-11-01']['user_id'])
next_day_users = set(data_audience[data_audience['date'] == '2023-11-02']['user_id'])
retention = first_day_users & next_day_users
retention_perc = len(retention) / len(first_day_users) * 100

print(f'{retention_perc:.2f} %')

26.65 %


In [ ]:
# 5. Во вкладке "Данные об аудитории" есть информация о том, сколько объявлений посмотрел каждый пользователь (view_adverts).
# Посчитайте пользовательскую конверсию в просмотр объявления за ноябрь? (в пользователях) 

In [112]:
all_users = data_audience['user_id'].nunique()
target_action_users = data_audience[data_audience['view_adverts'] != 0]['user_id'].nunique()
conversion_perc = target_action_users / all_users * 100

print(f'{conversion_perc:.2f} %')

46.31 %


In [ ]:
# 6. Используя информацию из вкладки "Данные об аудитории", посчитайте среднее количество просмотренных объявлений на пользователя в ноябре

In [113]:
total_view_adverts = data_audience['view_adverts'].sum()
all_users =  data_audience['user_id'].nunique()
average_view_per_user = total_view_adverts / all_users

print(f'{average_view_per_user:.2f}')

2.87


In [ ]:
# 7. Мы провели опрос среди 2000 пользователей. Из них 500 «критики», 1200 «сторонники» и 300 «нейтралы». Посчитайте, чему будет равен NPS

In [107]:
total = 2000
promoters_perc = 1200 / total * 100
detractors_perc = 500 / total * 100
nps = promoters_perc - detractors_perc
print(f'{nps:.2f} %')

35.00 %


In [ ]:
# 8. Во вкладке "Данные АБ-тестов" результаты трех несвязанных АБ тестов для ARPU (общая выручка/общее количество пользователей).
# Посмотрите на результаты тестов и интерпретируйте их. Напишите значения p-value, которые вы получили.
# Подготовьте выводы и рекомендации. 

# experiment_num - номер эксперимента
# experiment_group - группа, в которую попал пользователь
# user_id - id пользователя
# revenue - выручка, которую сгенерировал пользователь, купив платную услугу продвижения


In [11]:
data_AB = pd.read_excel('test_task.xlsx', sheet_name='Данные АБ тестов')
data_AB.head()

,experiment_num,experiment_group,user_id,revenue
0,1,test,38456,520
1,1,control,13125924,806
2,1,control,9761984,0
3,1,test,11387012,208
4,1,test,18319648,104


In [12]:
data_AB['experiment_num'].unique()

array([1, 2, 3])

In [13]:
data_exp_1_test = data_AB[(data_AB['experiment_num']==1) & (data_AB['experiment_group']=='test')]
data_exp_1_control = data_AB[(data_AB['experiment_num']==1) & (data_AB['experiment_group']=='control')]

arpu_agg_exp_1_test = data_exp_1_test.groupby('user_id').agg('sum')['revenue']
arpu_agg_exp_1_control = data_exp_1_control.groupby('user_id').agg('sum')['revenue']

stat, p_value = ttest_ind(arpu_agg_exp_1_test, arpu_agg_exp_1_control)

print(p_value, '\n')

if p_value < 0.05:
    print('Отвергаем Н0 - есть статистически значимая разница в средних значениях двух групп')
else:
    print('Не отвергаем Н0 - нет статистически значимой разницы в средних значениях двух групп')

0.688784211779017 

Не отвергаем Н0 - нет статистически значимой разницы в средних значениях двух групп


In [14]:
data_exp_2_test = data_AB[(data_AB['experiment_num']==2) & (data_AB['experiment_group']=='test')]
data_exp_2_control = data_AB[(data_AB['experiment_num']==2) & (data_AB['experiment_group']=='control')]

arpu_agg_exp_2_test = data_exp_2_test.groupby('user_id').agg('sum')['revenue']
arpu_agg_exp_2_control = data_exp_2_control.groupby('user_id').agg('sum')['revenue']

stat, p_value = ttest_ind(arpu_agg_exp_2_test, arpu_agg_exp_2_control)

print(p_value, '\n')

if p_value < 0.05:
    print('Отвергаем Н0 - есть статистически значимая разница в средних значениях двух групп')
else:
    print('Не отвергаем Н0 - нет статистически значимой разницы в средних значениях двух групп')

0.0009915972237576193 

Отвергаем Н0 - есть статистически значимая разница в средних значениях двух групп


In [15]:
data_exp_3_test = data_AB[(data_AB['experiment_num']==3) & (data_AB['experiment_group']=='test')]
data_exp_3_control = data_AB[(data_AB['experiment_num']==3) & (data_AB['experiment_group']=='control')]

arpu_agg_exp_3_test = data_exp_3_test.groupby('user_id').agg('sum')['revenue']
arpu_agg_exp_3_control = data_exp_3_control.groupby('user_id').agg('sum')['revenue']

stat, p_value = ttest_ind(arpu_agg_exp_3_test, arpu_agg_exp_3_control)

print(p_value, '\n')

if p_value < 0.05:
    print('Отвергаем Н0 - есть статистически значимая разница в средних значениях двух групп')
else:
    print('Не отвергаем Н0 - нет статистически значимой разницы в средних значениях двух групп')

0.06174290011218765 

Не отвергаем Н0 - нет статистически значимой разницы в средних значениях двух групп


In [ ]:
# 9. По датасету с листерами посчитайте средний доход на пользователя

In [114]:
data_listers = pd.read_excel('test_task.xlsx', sheet_name='Листеры')
data_listers.head()

,user_id,date,cnt_adverts,age,cnt_contacts,revenue
0,100,2022-01-01,6,21,119,53
1,100,2022-01-02,2,21,200,18
2,100,2022-01-03,6,21,193,42
3,100,2022-01-04,2,21,143,38
4,100,2022-01-05,2,21,190,40


In [121]:
total_revenue = data_listers['revenue'].sum()
all_users = data_listers['user_id'].nunique()

print(f'{total_revenue / all_users:.2f}')

156.48


In [ ]:
# 10. По датасету с листерами посчитайте медиану возраста пользователя

In [122]:
data_listers['age'].median()

28.0

In [ ]:
# 18. Были получены следующие результаты. Коллеги просят вас подтвердить их и сделать окончательный вывод по эксперименту.
# •	Вариант A (контрольная группа) — 100 047 501 посетитель, 1003 платежа.
# •	Вариант B (тестовая группа) — 100 001 055 посетителей, 1099 платежей.
# Какие рекомендации вы бы дали, основываясь на этих данных?

In [5]:
convertions = [1003, 1099]
visitors = [100047501, 100001055]
stat, p_value = proportions_ztest(convertions, visitors)

alpha = 0.05
if p_value < alpha:
    print('Отвергаем H0 - есть статистически значимая разница')
else:
    print('Не отвергаем H0 - разницы нет')

Отвергаем H0 - есть статистически значимая разница


In [10]:
control = convertions[0] / visitors[0]
test = convertions[1] / visitors[1]

share = (test - control) / control * 100
print(f'{share:.2f} %')

9.62 %
